# 3. Control Pulse Setup

## 3.1 First Transition: g-e

### 3.1.1 Amplitude Rabi Experiement

In [ ]:
# qubit drive frequency - defined in calibration on device setup as baseline reference
drive_Oscillator_q0.frequency = qubit_parameters["qb_freq"]
# set oscillator type to hardware to ensure optimal use of the instrument functionality
drive_Oscillator_q0.modulation_type = ModulationType.HARDWARE

In [ ]:
#Set power ranges for drive, measure and aquire
###USUALLY DON'T NEED TO BE USED!!!!!!

####Drive#### Usualy 5 dBm
lsg_q0["drive_line"].range = 10

####Measure###
#lsg_q0["measure_line"].range = -25a
# lsg_q0["measure_line"].range = -10 #High Power
lsg_q0["measure_line"].range = -20 #Low Power

####Acquire### Usualy -30
lsg_q0["acquire_line"].range = -35

In [ ]:
#Check ranges
print('Drive line range:', lsg_q0["drive_line"].range, 'dBm')
print('Measure line range:', lsg_q0["measure_line"].range, 'dBm')
print('Acquire line range:', lsg_q0["acquire_line"].range, 'dBm')

In [ ]:
range_params = QBaseParameters(sample=qsample_params,
                                    name=f'range_paramaters_cooldown_{cooldown_nr}',
                                    parameters={'drive': lsg_q0["drive_line"].range,
                                                'drive_ef': lsg_q0["drive_ef_line"].range,
                                                'measure': lsg_q0["measure_line"].range,
                                                'acquire': lsg_q0["acquire_line"].range})

range_params

In [ ]:
#Check frequencies
print('Readout frequency:', qubit_parameters["ro_freq"]*1e-6, 'MHz')
print('Qubit frequency:', qubit_parameters["qb_freq"]*1e-6, 'MHz')
# qubit_parameters.update_parameter("qb_freq", 599.5593508373827 * 1e6)

In [ ]:
#Set readout frequency

# readout pulse frequency resonant with the readout resonator
readout_Oscillator_q0.frequency = qubit_parameters["ro_freq"]
#readout_Oscillator_q0.frequency = qubit_parameters["ro_bare"]
# change oscillator type to software to enable multiplexed qubit readout
readout_Oscillator_q0.modulation_type = ModulationType.SOFTWARE

# demodulation frequency same as readout pulse frequency
acquire_Oscillator_q0.frequency = qubit_parameters["ro_freq"]
#acquire_Oscillator_q0.frequency = qubit_parameters["ro_bare"]
acquire_Oscillator_q0.modulation_type = ModulationType.SOFTWARE

In [ ]:
#change lenght of the qubit exitation pulse
qubit_parameters.update_parameter('qb_len', 300e-9)
#qubit_parameters.add_parameter('qb._len', 50e-9)

print('Qubit exitation pulse length: ', qubit_parameters["qb_len"]*1e9, 'ns')

In [ ]:
# qubit_parameters["relax"] = 7e-5
# qubit_parameters.update_parameter('relax',70e-6)

In [ ]:
readout_low = {'readout_type': 'pulse',
                'readout_pulse': readout_pulse,
                'readout_weighting_function': readout_weighting_function,
                'relax_time': qubit_parameters["relax"],
                'measure_freq': qubit_parameters["ro_freq"],
                'acquire_freq': qubit_parameters["ro_freq"],
                'readout_range': lsg_q0["measure_line"].range,
                'readout_delay': qubit_parameters['ro_int_delay']
               }

In [ ]:
readout_low

In [ ]:
# range of pulse amplitude scan
amp_min = 0
amp_max = 1.0
amp_num = 31

# set up sweep parameter - qubit drive pulse amplitude
rabi_sweep = LinearSweepParameter(
    uid="rabi_amp", start=amp_min, stop=amp_max, count=amp_num
)

# how many averages per point: 2^n_average
n_average = 11

# Rabi excitation pulse - gaussian of unit amplitude - amplitude will be scaled with sweep parameter in experiment
gaussian_pulse = pulse_library.gaussian(
    uid="gaussian_pulse", length=qubit_parameters["qb_len"], amplitude=1.0
)

In [ ]:
exp_rabi = create_rabi(rabi_sweep, 
                       gaussian_pulse, 
                       readout_low, #readout_opt, #readout_low,
                       n_average)

exp_rabi_comp = my_session.compile(exp_rabi)

In [ ]:
# run the experiment on qubit 0
my_results = my_session.run(exp_rabi_comp)

meas_ready()

In [ ]:
# get measurement data returned by the instruments
rabi_res = my_results.get_data("q0_rabi")

# define amplitude axis from qubit parameters
rabi_amp = my_results.get_axis("q0_rabi")[0]

In [ ]:
signal = rabi_res

fig, ax = plt.subplots(2, 2, sharex = True, figsize = (10, 8))

fig.suptitle('Rabi oscillations vs amplitude, $\Delta t_q$ = ' + '{:.0f}'.format(qubit_parameters["qb_len"]*1e9) + 'ns, '+'$N_{av}$ = ' + '{:.0f}'.format(n_average), fontsize=16)

fig.supxlabel("Amplitude (a.u.)")
fig.supylabel("Amplitude (a.u.)")

ax[0,0].plot(rabi_amp, abs(signal), ".k", label = 'amp')
ax[1,0].plot(rabi_amp, np.unwrap(np.angle(signal)), ".k", label = 'phase')
ax[0,1].plot(rabi_amp, np.real(signal), ".k", label = 'real')
ax[1,1].plot(rabi_amp, np.imag(signal), ".k", label = 'imag')

ax[0,0].set_title('amplitude')
ax[1,0].set_title('phase')
ax[0,1].set_title('real part')
ax[1,1].set_title('imaginary part')

amp_plot = np.linspace(rabi_amp[0], rabi_amp[-1], 5 * len(rabi_amp))

popt_amp, pcov_amp = fit_Rabi(rabi_amp, abs(signal), 10, 1, 1, 0.048, plot=False)
popt_pha, pcov_pha = fit_Rabi(rabi_amp, np.unwrap(np.angle(signal)), 10, 1, 1, 0.048, plot=False)
popt_real, pcov_real = fit_Rabi(rabi_amp, np.real(signal), 10, 1, 1, 0.048, plot=False)
popt_imag, pcov_imag = fit_Rabi(rabi_amp, np.imag(signal), 10, 1, 1, 0.048, plot=False)

ax[0,0].plot(amp_plot, func_osc(amp_plot, *popt_amp), "-r")
ax[1,0].plot(amp_plot, func_osc(amp_plot, *popt_pha), "-r")
ax[0,1].plot(amp_plot, func_osc(amp_plot, *popt_real), "-r")
ax[1,1].plot(amp_plot, func_osc(amp_plot, *popt_imag), "-r")

print('fitting results for x180-pulse amplitude:')
print('amp   --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_amp[0], np.sqrt(np.diag(pcov_amp))[0]))
print('pha   --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_pha[0], np.sqrt(np.diag(pcov_pha))[0]))
print('real  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_real[0], np.sqrt(np.diag(pcov_real))[0]))
print('imag  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_imag[0], np.sqrt(np.diag(pcov_imag))[0]))

#qubit_parameters["pi_amp"] = np.pi/popt_pha[0]
#qubit_parameters["pihalf_amp"] = np.pi / 2 / (popt_pha[0])

#save figure
figname = 'Rabi_osc_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
signal = rabi_res

fig, ax = plt.subplots(3, 1, sharex = True, figsize = (10, 8))

I_zero = np.real(signal)[0]
Q_zero = np.imag(signal)[0]

new_signal = np.sqrt((np.real(signal) - I_zero)**2 + (np.imag(signal) - Q_zero)**2)

fig.suptitle(r'Rabi oscillations vs amplitude, $\Delta t_q$ = ' + '{:.0f}'.format(qubit_parameters["qb_len"]*1e9) + 'ns, '+'$N_{av}$ = ' + '{:.0f}'.format(n_average), fontsize=16)

fig.supxlabel("Amplitude (a.u.)")
fig.supylabel("Amplitude (a.u.)")

ax[0].plot(rabi_amp, np.real(signal), ".k")
ax[1].plot(rabi_amp, np.imag(signal), ".k")
ax[2].plot(rabi_amp, new_signal, ".k")

ax[0].set_title('real')
ax[1].set_title('imag')
ax[2].set_title('norm')

amp_plot = np.linspace(rabi_amp[0], rabi_amp[-1], 5 * len(rabi_amp))

popt_real, pcov_real = fit_Rabi(rabi_amp, np.real(signal), 10, 1, 1, 0.048, plot=False)
popt_imag, pcov_imag = fit_Rabi(rabi_amp, np.imag(signal), 10, 1, 1, 0.048, plot=False)
popt_norm, pcov_norm = fit_Rabi(rabi_amp, new_signal, 10, 1, 1, 0.01, plot=False)

ax[0].plot(amp_plot, func_osc(amp_plot, *popt_real), "-r")
ax[1].plot(amp_plot, func_osc(amp_plot, *popt_imag), "-r")
ax[2].plot(amp_plot, func_osc(amp_plot, *popt_norm), "-r")

print(f"Current pi pulse amplitude: \n{qubit_parameters['pi_amp']}")
print('fitting results for x180-pulse amplitude:')
print('real  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_real[0], np.sqrt(np.diag(pcov_real))[0]))
print('imag  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_imag[0], np.sqrt(np.diag(pcov_imag))[0]))
print('norm  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_norm[0], np.sqrt(np.diag(pcov_norm))[0]))

#save figure
figname = 'Rabi_osc_norm_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
qubit_parameters.update_parameter("pi_amp", np.pi / (popt_norm[0]))
try:
    qubit_parameters.update_parameter("pihalf_amp", np.pi / 2 / (popt_norm[0]))
except Exception as ex:
    print(ex)
    qubit_parameters.add_parameter("pihalf_amp", np.pi / 2 / (popt_norm[0]))
    print('Added new parameter')

In [ ]:
#plot data on IQ plane
plt.title(r'Rabi oscillations on IQ, $\Delta t_q$ = ' + '{:.0f}'.format(qubit_parameters["qb_len"]*1e9) + 'ns, '+'$N_{av}$ = ' + '{:.0f}'.format(n_average), fontsize=16)

plt.plot(np.real(signal), np.imag(signal), ".k")

popt, pcov = fit_linear(np.real(signal), np.imag(signal), plot=False)

plt.plot(np.real(signal), func_lin(np.real(signal),*popt), 'r')

plt.xlabel('Real, a.u')
plt.ylabel('Imag, a.u')

In [ ]:
Data_rabi = {'rabi_res': rabi_res,
            'rabi_amp': rabi_amp
            }

Data_rabi.update(qubit_parameters._params)
Data_rabi.update(lo_settings._params)

file_name = 'Rabi_ampl_rough_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_rabi)

### 3.1.2 Set/Update x90 and x180 Pulses

print('Pi amplitude:', qubit_parameters["pi_amp"])
print('Pi length:', qubit_parameters["qb_len"])

In [ ]:
x180 = pulse_library.gaussian(
    uid="x180", length=qubit_parameters["qb_len"], amplitude=qubit_parameters["pi_amp"]
)

x180_drag = pulse_library.drag(
    uid="x180_drag", length=qubit_parameters["qb_len"], amplitude=qubit_parameters["pi_amp"], sigma=0.3, beta=0.2
)

x90 = pulse_library.gaussian(
    uid="x90",
    length=qubit_parameters["qb_len"],
    amplitude=qubit_parameters["pihalf_amp"],
)

### 3.1.3 Rabi Shevron

results_list = []

detuning_shev = np.linspace(-5, 5, 41)*1e6

In [ ]:
for i in range(len(detuning_shev)):
    print('Measurement number:', i)
    
    # qubit drive frequency - defined in calibration on device setup as baseline reference
    drive_Oscillator_q0.frequency = qubit_parameters["qb_freq"]+detuning_shev[i]
    # set oscillator type to hardware to ensure optimal use of the instrument functionality
    drive_Oscillator_q0.modulation_type = ModulationType.HARDWARE
    
    my_results = my_session.run(exp_rabi)
    results_list.append(my_results.get_data("q0_rabi"))

In [ ]:
rabi_shev_arr = np.array(results_list)

In [ ]:
X = rabi_amp
Y = detuning_shev*1e-6
#Z = np.unwrap(np.angle(rabi_shev_arr))
Z = np.absolute(rabi_shev_arr)
calc.plot_2d(Y, X,Z, flip = False, cmap = 'Reds')
plt.title('Rabi shevron, ABS, $\Delta t_q$ = ' + '{:.0f}'.format(qubit_parameters["qb_len"]*1e9) + 'ns, '+'$N_{av}$ = ' + '{:.0f}'.format(n_average), fontsize=16)
plt.xlabel('Detuning, MHz')
plt.ylabel('Rabi Amplitude')

#save figure
figname = 'Rabi_shevron_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
Data_rabi = {'rabi_shev': rabi_shev_arr,
               'detuning': detuning_shev,
               'rabi_amp': rabi_amp,
               }

Data_rabi.update(qubit_parameters._params)
Data_rabi.update(lo_settings._params)

file_name = 'Rabi_shev_ge_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_rabi)

In [ ]:
#Restore proper qubit frequency!

drive_Oscillator_q0.frequency = qubit_parameters["qb_freq"]
# set oscillator type to hardware to ensure optimal use of the instrument functionality
drive_Oscillator_q0.modulation_type = ModulationType.HARDWARE

### 3.1.4 Rabi Frequency Calibration

In [ ]:
#Settings for rabi frequency calibration
# range of pulse amplitude scan
amp_min = 0
amp_max = 1.0 #min([qubit_parameters["pi_amp"] * 2.2, 1.0])
# how many amplitude points to measure
amp_num = 101

# set up sweep parameter - qubit drive pulse amplitude
rabi_sweep = LinearSweepParameter(
    uid="rabi_amp", start=amp_min, stop=amp_max, count=amp_num
)

# how many averages per point: 2^n_average
n_average = 12

# # Rabi excitation pulse - gaussian of unit amplitude - amplitude will be scaled with sweep parameter in experiment
# gaussian_pulse = pulse_library.gaussian(
#     uid="gaussian_pulse", length=20e-9, amplitude=1.0
# )

In [ ]:
rabi_length_arr = np.linspace(50, 400, 36)*1e-9
print(rabi_length_arr)

In [ ]:
freq_test_list= [10]

rabi_list = []
rabi_popt_list = []
rabi_pcov_list = []

for i in range(len(rabi_length_arr)):
    print('Length: ', rabi_length_arr[i]*1e9, 'ns')
    gaussian_pulse_sw = pulse_library.gaussian(
        uid="gaussian_pulse", length=rabi_length_arr[i], amplitude=1.0
    )
    
    exp_rabi_sw = make_rabi(rabi_sweep, 
                         gaussian_pulse_sw, 
                         readout_pulse, 
                         readout_weighting_function, 
                         qubit_parameters["relax"], 
                         n_average)
    
    exp_rabi_sw.set_signal_map(qubit_meas_map)
    # compile
    exp_rabi_sw_comp = my_session.compile(exp_rabi_sw)
    
    rabi_sw_results = my_session.run(exp_rabi_sw_comp)
    
    # get measurement data returned by the instruments
    rabi_res = rabi_sw_results.get_data("q0_rabi")
    rabi_amp = rabi_sw_results.get_axis("q0_rabi")[0]
    
    I_zero = np.real(rabi_res)[0]
    Q_zero = np.imag(rabi_res)[0]
    
    norm_signal = np.sqrt((np.real(rabi_res) - I_zero)**2 + (np.imag(rabi_res) - Q_zero)**2)
    

    popt_norm, pcov_norm = fit_Rabi(rabi_amp, norm_signal, freq_test_list[-1], 1.0, 0.1, 0.048, plot=True)
    freq_test_list.append(popt_norm[0])
    
    rabi_list.append(rabi_res)
    rabi_popt_list.append(popt_norm)
    rabi_pcov_list.append(pcov_norm)
    
rabi_arr = np.array(rabi_list)
rabi_popt_arr = np.array(rabi_popt_list)
rabi_pcov_arr = np.array(rabi_pcov_list)

In [ ]:
X = rabi_amp
Y = rabi_length_arr
#Z = np.unwrap(np.angle(rabi_shev_arr))
Z = np.absolute(rabi_arr)
calc.plot_2d(Y, X,Z, flip = False, cmap = 'Reds')
plt.xlabel('Pulse_length')
plt.ylabel('Rabi_amplitude')

In [ ]:
#x = 1e-6*np.pi / rabi_length_arr
x = 1e-6/ rabi_length_arr
y = np.pi / (rabi_popt_arr[:,0])

popt, pcov = fit_linear(x, y, plot=False)

plt.plot(x, y, '.k')
plt.plot(x, func_lin(x,*popt), 'r')

plt.xlabel('Rabi frequency, MHz')
plt.ylabel('Rabi amplitude')

print('Slope: ', popt[0])
print('Intercept: ', popt[1])

qubit_parameters['rabi_slope'] = popt[0]
qubit_parameters['rabi_intercept'] = popt[1]

In [ ]:
Data_rabi_calib = {'rabi_length':rabi_length_arr,
                   'rabi_data': rabi_arr, 
                   'rabi_amp': rabi_amp,
                   'rabi_popt': rabi_popt_arr,
                   'rabi_pcov': rabi_pcov_arr,
                  }

Data_rabi_calib.update(qubit_parameters)
Data_rabi_calib.update(lo_settings)

file_name = 'Rabi_calib_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_rabi_calib)

### 3.1.5 T1 Experiment

In [ ]:
# qubit_parameters.update_parameter('relax', 500e-6)

In [ ]:
# delay range for ramsey pulses
t1_min = 0.0
t1_max = 100e-6 #5 * qubit_parameters["qb_t1"]
# how many delay points to sweep
t1_num = 21

# set up sweep parameter
#t1_sweep = LinearSweepParameter(uid="t1_delay", start=t1_min, stop=t1_max, count=t1_num)

#log sweep
def log_sweep_help(t1_min, t1_max, t1_num):
    t1_log_sweep = np.logspace(np.log10(t1_max/t1_num/10), np.log10(t1_max), t1_num-1)
    t1_log_sweep_arr = np.append([0.0], t1_log_sweep)
    return SweepParameter(values=t1_log_sweep_arr)

t1_sweep = log_sweep_help(t1_min, t1_max, t1_num)

# how many averages per point: 2^n_average
n_average = 12

In [ ]:
exp_t1 = create_t1(t1_sweep, 
                   x180, 
                   readout_low,#readout_opt
                   n_average)

exp_t1_comp = my_session.compile(exp_t1)

In [ ]:
# run the experiment on qubit 0
t1_results = my_session.run(exp_t1_comp)

meas_ready()

In [ ]:
# get measurement data returned by the instruments
t1_res = t1_results.get_data("q0_t1")

# define time axis from qubit parameters
t1_delay = t1_results.get_axis("q0_t1")[0]

In [ ]:
#fit the data
popt_sl, pcov_sl = auto_T1_fit(t1_delay, t1_res, data_type = 'rot', plot = True)

#Transform fit parameters to proper formatting for one slice measurements
popt = popt_sl[0,:]
pcov = pcov_sl[0,:,:]

plt.title('Decay, rotated data')
plt.figtext(0.6, 0.5, r'$T_1$ = '+'{:.2f}'.format(1/popt[0]*1e6)+r'$\mu$s')

# update qubit parameters - here relaxation time / qubit lifetime
qubit_parameters.update_parameter("qb_t1", 1 / popt[0])

# T1 error in us
err = np.sqrt(np.diag(pcov))
print(f'T1 = {qubit_parameters["qb_t1"]*1e6:.3f} +- {(err[0]/(popt[0]*popt[0])*1e6):.3f} us')

#save figure
figname = 'T1'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
Data_T1 = {'t1_res': t1_res,
            't1_delay': t1_delay
            }

Data_T1.update(qubit_parameters._params)
Data_T1.update(lo_settings._params)

file_name = 'T1_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_T1)

In [ ]:
#file_name = 'Qubit_params_dict_lower_ss_after_T1'
#file_path = get_path_to_file(file_name, '.txt', sample_parameters = sample_parameters)
#with open(file_path, 'w') as fp:
    #js.dump(qubit_parameters,fp)

### 3.1.6 Ramsey Experiments

#### A - One-slide Experiment

In [ ]:
#Set Ramsey detuning for single sweep
qubit_parameters.update_parameter("ramsey_det", 0.5e6)
qubit_parameters["ramsey_det"]

# qubit_parameters.update_parameter("relax", 100e-6)
#qubit_parameters.update_parameter('qb_t2_ramsey', 10e-6)

In [ ]:
qubit_parameters["ramsey_det"]

In [ ]:
# delay range for ramsey pulses
ramsey_min = 0.0

#ramsey_max = 5.0e-6 
ramsey_max = 15e-6 

# how many delay points to sweep
ramsey_num = 101
#ramsey_num = 151

# set up delay sweep parameter
ramsey_sweep = LinearSweepParameter(
    uid="ramsey_delay", start=ramsey_min, stop=ramsey_max, count=ramsey_num
)

# how many averages per point: 2^n_average
n_average = 12

In [ ]:
exp_ramsey = create_ramsey(ramsey_sweep, 
                           x90, 
                           readout_low, #readout_low,
                           n_average, 
                           qubit_parameters)

exp_ramsey_comp = my_session.compile(exp_ramsey)

In [ ]:
#my_results = my_session.run(exp_ramsey)
ramsey_results = my_session.run(exp_ramsey_comp)

meas_ready()

In [ ]:
# get measurement data returned by the instruments
ramsey_res = ramsey_results.get_data("q0_ramsey")

# define time axis from qubit parameters
ramsey_delay = ramsey_results.get_axis("q0_ramsey")[0]

In [ ]:
# get measurement data returned by the instruments
ramsey_res = ramsey_results.get_data("q0_ramsey")

# define time axis from qubit parameters
ramsey_delay = ramsey_results.get_axis("q0_ramsey")[0]

In [ ]:
# plot measurement results
fig = plt.figure(figsize=(12,8))

#data = ramsey_res.real
data = ramsey_res.imag
# data = abs(ramsey_res)
# data = np.unwrap(np.angle(ramsey_res))

plt.plot(ramsey_delay, data, ".k") #np.unwrap(np.angle(
#plt.plot(ramsey_delay, np.unwrap(np.angle(ramsey_res)), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("delay (s)")
plt.title('Ramsey oscillations')

# increase number of plot points for smooth plotting of fit results
delay_plot = np.linspace(ramsey_delay[0], ramsey_delay[-1], 5 * len(ramsey_delay))

## fit measurement data to decaying sinusoidal oscillatio
popt, pcov = fit_Ramsey(
    x=ramsey_delay,
    y=data,
    #np.unwrap(np.angle(ramsey_res)),
    freq=5* qubit_parameters["ramsey_det"],
    phase=0,
    #2 / qubit_parameters["qb_t2_ramsey"],
    rate=1 / 10e-6,
    amp=0.02,
    off=0.129,
    plot=False,
     bounds=[
         [0.1e6, -np.pi, 0.1 / qubit_parameters["qb_t2_ramsey"], 0.0001, -2],
         [50e6, np.pi, 10 / qubit_parameters["qb_t2_ramsey"], 2, 2],
    ],
#    bounds=[
#        [0.01e6, -1.5*np.pi, 0.05 / 6e-5, 0.0001, -4],
#        [150e6, 1.5*np.pi, 30 / 6e-5, 10, 10],
#    ],
)
#print(popt)

# plot fit results together with experimental data
plt.plot(delay_plot, func_decayOsc(delay_plot, *popt), "-r")
plt.figtext(0.6, 0.8, 'T2 = '+'{:.3f}'.format(1/popt[2]*1e6)+'us')
plt.figtext(0.6, 0.75, r'$\Delta$ = '+'{:.3f}'.format(popt[0]/(2*np.pi)*1e-6)+'MHz')

# update qubit parameters - here qubit dephasing time and qubit frequency
#qubit_parameters["qb_t2_ramsey"] = 1 / popt[2]
# qubit_parameters.update_parameter('qb_t2_ramsey', 1 / popt[2])
# qubit_parameters.update_parameter('qb_freq', qubit_parameters['qb_freq'] + qubit_parameters["ramsey_det"] - popt[0]/np.pi/2) #for positive detuning
# qubit_parameters.update_parameter('qb_freq', qubit_parameters['qb_freq'] + qubit_parameters["ramsey_det"] + popt[0]/np.pi/2) #for negative detuning

# T2 Ramsey error in us
err = np.sqrt(np.diag(pcov))
print(f'T2* = {1/popt[2]*1e6:.3f} +- {err[2]/popt[2]*qubit_parameters["qb_t2_ramsey"]*1e6:.3f} us')
print(f'Detuning is {popt[0]/np.pi/2*1e-6:.4} +- {err[0]/np.pi/2*1e-6:.2} MHz')

#save figure
figname = 'Ramsey_slice_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
qb_f_neg=qubit_parameters['qb_freq'] + qubit_parameters["ramsey_det"] - popt[0]/2/np.pi
qb_f_pos=qubit_parameters['qb_freq'] - qubit_parameters["ramsey_det"] + popt[0]/2/np.pi

In [ ]:
#try pos
qubit_parameters.update_parameter('qb_freq',qb_f_pos)

In [ ]:
#qubit_parameters.update_parameter('qb_freq',47.70797100737143*1e6) #default

In [ ]:
qubit_parameters['qb_freq']

In [ ]:
#try neg
qubit_parameters.update_parameter('qb_freq',qb_f_neg)

#### B - Ramsey Fringes

In [ ]:
#set ramsey detuning limitst and point number

ramsey_min_freq = -2
ramsey_max_freq = 2
N_sweep = 21

ramsey_shev_det = np.linspace(ramsey_min_freq, ramsey_max_freq, N_sweep)*1e6

In [ ]:
ramsey_sh_list = []

for i in range(N_sweep):
    ramsey_cal_sh = make_ramsey_calib(qubit_parameters["qb_freq"], ramsey_shev_det[i])
    
    exp_ramsey.set_calibration(ramsey_cal_sh)
    shev_results = my_session.run(exp_ramsey)
    
    ramsey_sh_list.append(shev_results.get_data("q0_ramsey"))

ramsey_fringes_arr = np.array(ramsey_sh_list)

#restore proper calibration
exp_ramsey.set_calibration(ramsey_cal)

In [ ]:
plt.plot(ramsey_delay, np.unwrap(np.angle(ramsey_fringes_arr[25,:])))

In [ ]:
ramsey_fringes_arr.shape

In [ ]:
X = ramsey_shev_det
Y = ramsey_delay
Z = np.absolute(ramsey_fringes_arr)
#Z = np.unwrap(np.angle(ramsey_fringes_arr))
calc.plot_2d(X, Y, Z, flip = False)

# figname = 'Ramsey_fringes_'
# file_path = get_path_to_file(figname, '.png')
# plt.savefig(file_path, dpi=600, format='png', metadata=None,
#         bbox_inches=None, pad_inches=0.1,
#         facecolor='auto', edgecolor='auto',
#         backend=None,
#        )

In [ ]:
Data_ramsey_shev = {'ramsey_fr': ramsey_fringes_arr,
                   'ramsey_delay': ramsey_delay, 
                    'ramsey_fr_det': ramsey_shev_det
               }

Data_ramsey_shev.update(qubit_parameters)
Data_ramsey_shev.update(lo_settings)

file_name = 'Ramsey_fringes_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_ramsey_shev)

### 3.1.7 Hahn Echo Experiment

In [ ]:
# delay range for echo pulses
echo_min = 0.0
echo_max = 50e-6 # 5.0 * qubit_parameters["qb_t2_echo"]
# how many delay points to sweep
echo_num = 151

# # set up delay sweep parameter - max of sweep parameter is half of max delay since here are two delays in sequence
# echo_sweep = LinearSweepParameter(
#     uid="echo_delay", start=echo_min, stop=echo_max / 2, count=echo_num
# )

echo_sweep = log_sweep_help(echo_min, echo_max/2, echo_num)

# how many averages per point: 2^n_average
n_average = 12

In [ ]:
echo_sweep.stop = echo_max

In [ ]:
exp_echo = create_echo(echo_sweep, 
                       x90, 
                       x180, 
                       readout_low, 
                       n_average)

exp_echo_comp = my_session.compile(exp_echo)

In [ ]:
echo_results = my_session.run(exp_echo_comp)

In [ ]:
# get measurement data returned by the instruments
echo_res = echo_results.get_data("q0_echo")

# define time axis from qubit parameters
echo_delay = echo_results.get_axis("q0_echo")[0]

In [ ]:
# plot measurement results - multiply delay by factor of two to get full sequence length
fig = plt.figure()
#plt.plot(2 * echo_delay, abs(echo_res), ".k") #np.unwrap(np.angle(
plt.plot(2 * echo_delay, np.unwrap(np.angle(echo_res)), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("delay (s)")
plt.title('Hahn echo')

# increase number of plot points for smooth plotting of fit results
delay_plot = np.linspace(echo_delay[0], 2 * echo_delay[-1], 5 * len(echo_delay))

## fit measurement data to decaying sinusoidal oscillatio
#popt, pcov = fit_T1(2 * echo_delay, abs(echo_res), 1e6, 0, -1, plot=False)
popt, pcov = fit_T1(2 * echo_delay, np.unwrap(np.angle(echo_res)), 1e6, 0, -1, plot=False)
print(popt)

# plot fit results together with experimental data
plt.plot(delay_plot, func_exp(delay_plot, *popt), "-r")

# update qubit parameters - here qubit dephasing time
# qubit_parameters["qb_t2_echo"] = 1 / popt[0]
print(1 / popt[0] * 1e6, ' us')

# T2 Echo error in us
err = np.sqrt(np.diag(pcov))
#print(f'T2E = {qubit_parameters["qb_t2_echo"]*1e6:.3f} +- {err[2]/popt[2]*qubit_parameters["qb_t2_echo"]*1e6:.3f} us')

#Save figure
figname = 'Hahn_echo'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
Data_hahn_echo = {'echo_res': echo_res,
               'echo_delay': echo_delay
               }

Data_hahn_echo.update(qubit_parameters)
Data_hahn_echo.update(lo_settings)

file_name = 'Hahn_echo_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_hahn_echo)

In [ ]:
file_name = 'Qubit_params_dict'
file_path = get_path_to_file(file_name, '.txt', sample_parameters)
with open(file_path, 'w') as fp:
    js.dump(qubit_parameters,fp)

### 3.1.8 Rabi Error Amplification

#### A - x90 Pulse

In [ ]:
max_pulses = 15
n_average = 12

rabi_err_amp_flips = np.arange(1, max_pulses+1, 2)
flips_sweep = SweepParameter(values=rabi_err_amp_flips)

print(f'Repition sequence: {rabi_err_amp_flips}')
print(f'Approximate length of longest pulse sequence: {rabi_err_amp_flips.max() * x90.length * 1e6} us')

# qubit_parameters.update_parameter('relax',  200e-6)
# readout_opt['relax_time'] = qubit_parameters['relax']

In [ ]:
exp_rabi_error_amp_eg_x90 = create_rabi_error_amp(flips_sweep, x180, x90, 
                                              readout_low,
                                              n_average,
                                             target_pulse='x90eg')

exp_rabi_error_amp_eg_comp_x90 = my_session.compile(exp_rabi_error_amp_eg_x90)

# show_pulse_sheet("Rabi Error Amplification", exp_rabi_error_amp_eg_comp)

In [ ]:
rabi_err_amp_results_x90 = my_session.run(exp_rabi_error_amp_eg_comp_x90)

In [ ]:
# get measurement data returned by the instruments
rabi_err_amp_res = rabi_err_amp_results_x90.get_data("q0_rabi")
rabi_err_amp_res_0 = rabi_err_amp_results_x90.get_data("calib_0")
rabi_err_amp_res_1 = rabi_err_amp_results_x90.get_data("calib_1")

In [ ]:
fig, ax = plt.subplots()
amp = np.abs(rabi_err_amp_res)

ax.scatter(rabi_err_amp_flips-1, amp)

ax.set_ylabel('Amplitude (a.u.)')
ax.set_xlabel('Number of applied pi/2 pulses')

axa = ax.twiny()
axa.set_xlabel('Approximate Pulse Sequence Duration (us)')
axa.plot(rabi_err_amp_flips * x90.length * 1e6, np.abs(rabi_err_amp_res), alpha=0)

plot_flips = np.linspace(1, max_pulses + 1, 501)
print(rabi_err_amp_flips)

calib_amp = np.abs(rabi_err_amp_res_0 - rabi_err_amp_res_1) / 2

try:
    popt_amp, pcov_amp = fit_osc_fixed_amp_phase(rabi_err_amp_flips - 1, amp, 
                                  freq=1.5, 
                                  fixed_phase=np.pi / 2, 
                                  fixed_amp=calib_amp, 
                                  off=np.mean(amp))
    popt_amp = list(popt_amp)
    popt_amp.insert(1, np.pi / 2)
    popt_amp.insert(2, calib_amp)
    ax.plot(plot_flips - 1, func_osc(plot_flips - 1 , *popt_amp), "-r")
    # print(popt_amp)

except RuntimeError as ex:
    print('ERROR!')
    print(ex, end='\n\n')
    
ax.hlines([np.abs(rabi_err_amp_res_0)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dashed')
ax.hlines([np.abs(rabi_err_amp_res_1)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dotted')

amp_corr = 0.5 * np.pi / popt_amp[0]
print(f'Current x90 amplitude: {qubit_parameters["pihalf_amp"]}')
print(f'Amplitude correction factor: {amp_corr}')

#### B - Update x90 Pulse

In [ ]:
print(f'Old x90 amplitude: {qubit_parameters["pihalf_amp"]}')
qubit_parameters.update_parameter("pihalf_amp", qubit_parameters["pihalf_amp"] * amp_corr)
print(f'New x90 amplitude: {qubit_parameters["pihalf_amp"]}')

In [ ]:
amp_corr

In [ ]:
x90 = pulse_library.gaussian(
    uid="x90", length=qubit_parameters["qb_len"], amplitude=qubit_parameters["pihalf_amp"]
)

#### C - x180 Pulse

In [ ]:
#qubit_parameters.update_parameter("pihalf_amp", qubit_parameters["pi_amp"]/2)

In [ ]:
max_pulses = 15
n_average = 12

rabi_err_amp_flips = np.arange(0, max_pulses+1)
flips_sweep = SweepParameter(values=rabi_err_amp_flips)

print(f'Repition sequence: {rabi_err_amp_flips}')
print(f'Approximate length of longest pulse sequence: {(rabi_err_amp_flips.max() * x180.length + x90.length) * 1e6} us')

# qubit_parameters.update_parameter('relax',  150e-6)
# readout_opt['relax_time'] = qubit_parameters['relax']

In [ ]:
#qubit_parameters.update_parameter('relax',  400e-6)

In [ ]:
exp_rabi_error_amp_eg = create_rabi_error_amp(flips_sweep, x180, x90, 
                                              readout_low,
                                              n_average,
                                             target_pulse='x180eg')

exp_rabi_error_amp_eg_comp = my_session.compile(exp_rabi_error_amp_eg)

# show_pulse_sheet("Rabi Error Amplification", exp_rabi_error_amp_eg_comp)

In [ ]:
rabi_err_amp_results = my_session.run(exp_rabi_error_amp_eg_comp)

In [ ]:
# get measurement data returned by the instruments
rabi_err_amp_res = rabi_err_amp_results.get_data("q0_rabi")
rabi_err_amp_res_0 = rabi_err_amp_results.get_data("calib_0")
rabi_err_amp_res_1 = rabi_err_amp_results.get_data("calib_1")

In [ ]:
fig, ax = plt.subplots()

ax.scatter(rabi_err_amp_flips, np.abs(rabi_err_amp_res))

ax.set_ylabel('Amplitude (a.u.)')
ax.set_xlabel('Number of applied pi pulses')

axa = ax.twiny()
axa.set_xlabel('Approximate Pulse Sequence Duration (us)')
axa.plot((rabi_err_amp_flips * x180.length + x90.length) * 1e6, np.abs(rabi_err_amp_res), alpha=0)

plot_flips = np.linspace(0, max_pulses + 1, 501)

amp = np.abs(rabi_err_amp_res)
calib_amp = np.abs(rabi_err_amp_res_0 - rabi_err_amp_res_1) / 2

try:
    popt_amp, pcov_amp = fit_osc_fixed_amp_phase(rabi_err_amp_flips, amp, 
                                  freq=np.pi, 
                                  fixed_phase=np.pi / 2, 
                                  fixed_amp=calib_amp, 
                                  off=np.mean(amp))
    popt_amp = list(popt_amp)
    popt_amp.insert(1, np.pi / 2)
    popt_amp.insert(2, calib_amp)
    ax.plot(plot_flips, func_osc(plot_flips, *popt_amp), "-r")

except RuntimeError as ex:
    print('ERROR!')
    print(ex, end='\n\n')
    
ax.hlines([np.abs(rabi_err_amp_res_0)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dashed')
ax.hlines([np.abs(rabi_err_amp_res_1)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dotted')

amp_corr = np.pi / popt_amp[0]
print(f'Current x180 amplitude: {qubit_parameters["pi_amp"]}')
print(f'Amplitude correction factor: {amp_corr}')

#### D - Update x180 Pulse

In [ ]:
print(f'Old x180 amplitude: {qubit_parameters["pi_amp"]}')
qubit_parameters.update_parameter("pi_amp", qubit_parameters["pi_amp"] * amp_corr)
print(f'New x180 amplitude: {qubit_parameters["pi_amp"]}')

In [ ]:
x180 = pulse_library.gaussian(
    uid="x180", length=qubit_parameters["qb_len"], amplitude=qubit_parameters["pi_amp"]
)

## 3.2 Second Transition: e-f

### 3.2.1 Amplitude Rabi Experiment

In [ ]:
# range of pulse amplitude scan
amp_min = 0
amp_max = 1.0 #min([qubit_parameters["pi_amp"] * 2.2, 1.0])
# how many amplitude points to measure
amp_num = 51

# set up sweep parameter - qubit drive pulse amplitude
rabi_ef_sweep = LinearSweepParameter(
    uid="rabi_ef_amp", start=amp_min, stop=amp_max, count=amp_num
)

# qubit_parameters.add_parameter("qb_ef_len", 50e-9)
qubit_parameters.update_parameter("qb_ef_len", 300e-9)
# how many averages per point: 2^n_average
n_average = 12

In [ ]:
#qubit_parameters.update_parameter("qb_ef_freq", -0.16561999999999966*1e6)

In [ ]:
qubit_parameters["qb_ef_len"]

In [ ]:
gaussian_ef = pulse_library.gaussian(
    uid="gaussian_ef", length=qubit_parameters["qb_ef_len"], amplitude=1.0
)

#qubit_parameters["qb_len"]

In [ ]:
# qubit_parameters['qb_ef_freq'] = 1
#qubit_parameters.update_parameter('qb_ef_freq', qubit_parameters["qb_freq"] - qubit_parameters['qb_anharm'])
print('Qubit ef transition frequency: ', qubit_parameters['qb_ef_freq']*1e-6, 'MHz')

In [ ]:
detuning_ef = 0 #e6

# qubit drive frequency - defined in calibration on device setup as baseline reference
drive_ef_Oscillator_q0.frequency = qubit_parameters["qb_ef_freq"] + detuning_ef
# set oscillator type to hardware to ensure optimal use of the instrument functionality
drive_ef_Oscillator_q0.modulation_type = ModulationType.HARDWARE

In [ ]:
exp_rabi_ef = create_rabi_ef(rabi_ef_sweep, 
                             gaussian_ef, 
                             readout_opt,
                             #readout_low,
                             n_average, 
                             x180, 
                             ge_amp = 1.0, 
                             n_ge_pulses = 2)

exp_rabi_ef_comp = my_session.compile(exp_rabi_ef)

show_pulse_sheet("Rabi-ef", exp_rabi_ef_comp)

In [ ]:
#my_results = my_session.run(exp_rabi_ef_comp)

rabi_ef_results = my_session.run(exp_rabi_ef_comp)

In [ ]:
# get measurement data returned by the instruments
rabi_ef_res = rabi_ef_results.get_data("q0_rabi_ef")

# define amplitude axis from qubit parameters
rabi_ef_amp = rabi_ef_results.get_axis("q0_rabi_ef")[0]

In [ ]:
# plot measurement data
fig = plt.figure()
plt.plot(rabi_ef_amp, abs(rabi_ef_res), ".k")
plt.ylabel("A (a.u.)")
# plt.plot(rabi_ef_amp, np.unwrap(np.angle(rabi_ef_res)), ".k")
# plt.ylabel("phase (deg)")
plt.xlabel("amplitude (a.u.)")
plt.title('Rabi oscillations for ef-transition')
#ylim = [0.0370, 0.0400]
#plt.ylim(ylim)

# increase number of plot points for smooth plotting of fit results
ef_amp_plot = np.linspace(rabi_ef_amp[0], rabi_ef_amp[-1], 5 * len(rabi_ef_amp))

# fit measurement results - assume sinusoidal oscillation with drive amplitude
popt, pcov = fit_Rabi(rabi_ef_amp, abs(rabi_ef_res), 20, 1.0, 1.0, 0.1626, plot=False)
# popt, pcov = fit_Rabi(rabi_ef_amp, np.unwrap(np.angle(rabi_ef_res)), 10, 1.0, 0.1, -1.9, plot=False)

# plot fit results together with measurement data
plt.plot(ef_amp_plot, func_osc(ef_amp_plot, *popt), "-r")
#plt.plot(ef_amp_plot, func_decayOsc(ef_amp_plot, *popt), "-r")
# update qubit parameters - pulse amplitdues for pi and pi/2 pulses
# qubit_parameters.update_parameter("pi_ef_amp", np.pi / (popt[0]))
print(qubit_parameters["pi_ef_amp"])

#qubit_parameters["pihalf_ef_amp"] = np.pi / 2 / (popt[0])
#print(qubit_parameters["pihalf_ef_amp"])

print(np.pi / (popt[0]))

#Nmin = np.argmax(func_decayOsc(ef_amp_plot, *popt))
#print(ef_amp_plot[Nmin])

# qubit_parameters["pi_ef_amp"] = ef_amp_plot[Nmin]
# print(qubit_parameters["pi_ef_amp"])

# qubit_parameters["pihalf_ef_amp"] = ef_amp_plot[Nmin] / 2
# print(qubit_parameters["pihalf_ef_amp"])

In [ ]:
signal = rabi_ef_res

fig, ax = plt.subplots(3, 1, sharex = True, figsize = (10, 8))

I_zero = np.real(signal)[0]
Q_zero = np.imag(signal)[0]

new_signal = np.sqrt((np.real(signal) - I_zero)**2 + (np.imag(signal) - Q_zero)**2)

fig.suptitle('Rabi oscillations vs pulse amplitude', fontsize=16)

fig.supxlabel("Amplitude (a.u.)")
fig.supylabel("Amplitude (a.u.)")

ax[0].plot(rabi_ef_amp, np.real(signal), ".k")
ax[1].plot(rabi_ef_amp, np.imag(signal), ".k")
ax[2].plot(rabi_ef_amp, new_signal, ".k")

ax[0].set_title('real part')
ax[1].set_title('imaginary part')
ax[2].set_title('norm')

amp_ef_plot = np.linspace(rabi_ef_amp[0], rabi_ef_amp[-1], 5 * len(rabi_ef_amp))

popt_real, pcov_real = fit_Ramsey(rabi_ef_amp, np.real(signal),
    30, 0, 2,
    amp=0.01, off=2.16, plot=False)
popt_imag, pcov_imag = fit_Ramsey(rabi_ef_amp, np.imag(signal),
    30, 0, 2,
    amp=0.001, off=0.45, plot=False)
popt_norm, pcov_norm = fit_Ramsey(rabi_ef_amp[:], new_signal[:],
    30, 0, 2,
    amp=0.001, off=2e-3, plot=False)

ax[0].plot(amp_ef_plot, func_decayOsc(amp_ef_plot, *popt_real), "-r")
ax[1].plot(amp_ef_plot, func_decayOsc(amp_ef_plot, *popt_imag), "-r")
ax[2].plot(amp_ef_plot, func_decayOsc(amp_ef_plot, *popt_norm), "-r")


print('fitting results for x180-pulse amplitude:')
print('real  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_real[0], np.pi*np.sqrt(np.diag(pcov_real))[0]/(popt_real[0])**2))
print('imag  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_imag[0], np.pi*np.sqrt(np.diag(pcov_imag))[0]/(popt_imag[0])**2))
print('norm  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_norm[0], np.pi*np.sqrt(np.diag(pcov_norm))[0]/(popt_norm[0])**2))

Nmax = np.argmax(func_decayOsc(amp_ef_plot, *popt_norm))
print(amp_ef_plot[Nmax])

# qubit_parameters["pi_ef_amp"] = amp_ef_plot[Nmax]
# qubit_parameters["pihalf_ef_amp"] = amp_ef_plot[Nmax] / 2
# print(qubit_parameters["pi_ef_amp"])
# print(qubit_parameters["pihalf_ef_amp"])

#qubit_parameters.update_parameter("pi_ef_amp", np.pi/popt_norm[0])
print(qubit_parameters["pi_ef_amp"])

#qubit_parameters.update_parameter("pihalf_ef_amp", np.pi / 2 / (popt_norm[0]))
print(qubit_parameters["pihalf_ef_amp"])

# qubit_parameters["pi_ef_amp"] = np.pi/popt_real[0]
#print(qubit_parameters["pi_ef_amp"])

# qubit_parameters["pihalf_ef_amp"] = np.pi / 2 / (popt_real[0])
# print(qubit_parameters["pihalf_ef_amp"])

In [ ]:
#qubit_parameters.add_parameter("pi_ef_amp", np.pi/popt_norm[0])
#qubit_parameters.add_parameter("pihalf_ef_amp", np.pi / 2 / (popt_norm[0]))

In [ ]:
#set a pi_ef pulse
x180_ef = pulse_library.gaussian(
    uid="x180_ef", length=qubit_parameters["qb_ef_len"], amplitude=qubit_parameters["pi_ef_amp"]
)

#set a pi_ef pulse
x90_ef = pulse_library.gaussian(
    uid="x90_ef", length=qubit_parameters["qb_ef_len"], amplitude=qubit_parameters["pi_ef_amp"]/2
)

### 3.2.2 Rabi Error Amplification

#### A - x90 Pulse

In [ ]:
max_pulses = 19
n_average = 13

rabi_err_amp_flips = np.arange(1, max_pulses+1, 2)
flips_sweep = SweepParameter(values=rabi_err_amp_flips)

print(f'Repition sequence: {rabi_err_amp_flips}')
print(f'Approximate length of longest pulse sequence: {(rabi_err_amp_flips.max() * x90_ef.length + 2 * x180.length) * 1e6} us')

# qubit_parameters.update_parameter('relax',  200e-6)
# readout_opt['relax_time'] = qubit_parameters['relax']

In [ ]:
exp_rabi_error_amp_ef_x90 = create_rabi_error_amp_ef(flips_sweep, x180, x90, x180_ef, x90_ef,
                                              readout_opt,
                                              n_average,
                                             target_pulse='x90ef')

exp_rabi_error_amp_ef_comp_x90 = my_session.compile(exp_rabi_error_amp_ef_x90)

# show_pulse_sheet("Rabi Error Amplification ef", exp_rabi_error_amp_ef_comp_x90)

In [ ]:
rabi_err_amp_results_x90_ef = my_session.run(exp_rabi_error_amp_ef_comp_x90)

In [ ]:
# get measurement data returned by the instruments
rabi_err_amp_res = rabi_err_amp_results_x90_ef.get_data("q0_rabi")
rabi_err_amp_res_0 = rabi_err_amp_results_x90_ef.get_data("calib_0")
rabi_err_amp_res_1 = rabi_err_amp_results_x90_ef.get_data("calib_1")

In [ ]:
fig, ax = plt.subplots()
amp = np.abs(rabi_err_amp_res)

ax.scatter(rabi_err_amp_flips-1, amp)

ax.set_ylabel('Amplitude (a.u.)')
ax.set_xlabel('Number of applied pi/2 pulses')

axa = ax.twiny()
axa.set_xlabel('Approximate Pulse Sequence Duration (us)')
axa.plot((rabi_err_amp_flips * x90_ef.length + 2 * x180.length) * 1e6, np.abs(rabi_err_amp_res), alpha=0)

plot_flips = np.linspace(1, max_pulses + 1, 501)
print(rabi_err_amp_flips)

calib_amp = np.abs(rabi_err_amp_res_0 - rabi_err_amp_res_1) / 2

try:
    popt_amp, pcov_amp = fit_osc_fixed_amp_phase(rabi_err_amp_flips - 1, amp, 
                                  freq=1.5, 
                                  fixed_phase=np.pi / 2, 
                                  fixed_amp=calib_amp, 
                                  off=np.mean(amp))
    popt_amp = list(popt_amp)
    popt_amp.insert(1, np.pi / 2)
    popt_amp.insert(2, calib_amp)
    ax.plot(plot_flips - 1, func_osc(plot_flips - 1 , *popt_amp), "-r")
    # print(popt_amp)

except RuntimeError as ex:
    print('ERROR!')
    print(ex, end='\n\n')
    
ax.hlines([np.abs(rabi_err_amp_res_0)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dashed')
ax.hlines([np.abs(rabi_err_amp_res_1)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dotted')

amp_corr = 0.5 * np.pi / popt_amp[0]
print(f'Current x90 ef amplitude: {qubit_parameters["pihalf_ef_amp"]}')
print(f'Amplitude correction factor: {amp_corr}')

#### B - Update x90 Pulse

In [ ]:
print(f'Old x90 ef amplitude: {qubit_parameters["pihalf_ef_amp"]}')
qubit_parameters.update_parameter("pihalf_ef_amp", qubit_parameters["pihalf_ef_amp"] * amp_corr)
print(f'New x90 ef amplitude: {qubit_parameters["pihalf_ef_amp"]}')

In [ ]:
x90_ef = pulse_library.gaussian(
    uid="x90_ef", length=qubit_parameters["qb_ef_len"], amplitude=qubit_parameters["pihalf_ef_amp"]
)

#### C - x180 Pulse

In [ ]:
max_pulses = 25
n_average = 12

rabi_err_amp_flips = np.arange(0, max_pulses+1)
flips_sweep = SweepParameter(values=rabi_err_amp_flips)

print(f'Repition sequence: {rabi_err_amp_flips}')
print(f'Approximate length of longest pulse sequence: {(rabi_err_amp_flips.max() * x180_ef.length + 2 * x180.length + x90_ef.length) * 1e6} us')

# qubit_parameters.update_parameter('relax',  150e-6)
# readout_opt['relax_time'] = qubit_parameters['relax']

In [ ]:
exp_rabi_error_amp_ef = create_rabi_error_amp_ef(flips_sweep, x180, x90, x180_ef, x90_ef,
                                              readout_opt,
                                              n_average,
                                             target_pulse='x180ef')

exp_rabi_error_amp_ef_comp = my_session.compile(exp_rabi_error_amp_ef)

# show_pulse_sheet("Rabi Error Amplification", exp_rabi_error_amp_eg_comp)

In [ ]:
rabi_err_amp_results = my_session.run(exp_rabi_error_amp_ef_comp)

In [ ]:
# get measurement data returned by the instruments
rabi_err_amp_res = rabi_err_amp_results.get_data("q0_rabi")
rabi_err_amp_res_0 = rabi_err_amp_results.get_data("calib_0")
rabi_err_amp_res_1 = rabi_err_amp_results.get_data("calib_1")

In [ ]:
fig, ax = plt.subplots()

ax.scatter(rabi_err_amp_flips, np.abs(rabi_err_amp_res))

ax.set_ylabel('Amplitude (a.u.)')
ax.set_xlabel('Number of applied pi pulses')

axa = ax.twiny()
axa.set_xlabel('Approximate Pulse Sequence Duration (us)')
axa.plot((rabi_err_amp_flips * x180_ef.length + 2 * x180.length + x90_ef.length) * 1e6, np.abs(rabi_err_amp_res), alpha=0)

plot_flips = np.linspace(0, max_pulses + 1, 501)

amp = np.abs(rabi_err_amp_res)
calib_amp = np.abs(rabi_err_amp_res_0 - rabi_err_amp_res_1) / 2

try:
    popt_amp, pcov_amp = fit_osc_fixed_amp_phase(rabi_err_amp_flips, amp, 
                                  freq=np.pi, 
                                  fixed_phase=np.pi / 2, 
                                  fixed_amp=calib_amp, 
                                  off=np.mean(amp))
    popt_amp = list(popt_amp)
    popt_amp.insert(1, np.pi / 2)
    popt_amp.insert(2, calib_amp)
    ax.plot(plot_flips, func_osc(plot_flips, *popt_amp), "-r")

except RuntimeError as ex:
    print('ERROR!')
    print(ex, end='\n\n')
    
ax.hlines([np.abs(rabi_err_amp_res_0)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dashed')
ax.hlines([np.abs(rabi_err_amp_res_1)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dotted')

amp_corr = np.pi / popt_amp[0]
print(f'Current x180 ef amplitude: {qubit_parameters["pi_ef_amp"]}')
print(f'Amplitude correction factor: {amp_corr}')

#### D - Update x180 Pulse

In [ ]:
print(f'Old x180 ef amplitude: {qubit_parameters["pi_ef_amp"]}')
qubit_parameters.update_parameter("pi_ef_amp", qubit_parameters["pi_ef_amp"] * amp_corr)
print(f'New x180 ef amplitude: {qubit_parameters["pi_ef_amp"]}')

In [ ]:
x180_ef = pulse_library.gaussian(
    uid="x180_ef", length=qubit_parameters["qb_ef_len"], amplitude=qubit_parameters["pi_ef_amp"]
)

### 3.2.3 Rabi Shevron

In [ ]:
results_list = []

detuning_ef = np.linspace(-20.0, 20.0, 31)*1e6

In [ ]:
(qubit_parameters["qb_ef_freq"] + lo_settings['qb_lo']) * 1e-9

In [ ]:
for i in range(len(detuning_ef)):
    print('Measurement number:', i)
    # qubit drive frequency - defined in calibration on device setup as baseline reference
    drive_ef_Oscillator_q0.frequency = qubit_parameters["qb_ef_freq"] + detuning_ef[i]
    # set oscillator type to hardware to ensure optimal use of the instrument functionality
    drive_ef_Oscillator_q0.modulation_type = ModulationType.HARDWARE
    
    my_results = my_session.run(exp_rabi_ef)
    results_list.append(my_results.get_data("q0_rabi_ef"))
    
results_arr = np.array(results_list)

In [ ]:
i = 15
#plt.plot(rabi_ef_amp, np.unwrap(np.angle(results_list[11])), "-o")
plt.plot(rabi_ef_amp, results_list[i].imag, "-o")
#plt.plot(rabi_ef_amp, np.unwrap(np.angle(results_list[23])), "-o")
print(detuning_ef[i]*1e-6, 'MHz')

In [ ]:
X = rabi_ef_amp
Y = detuning_ef
# Z = np.unwrap(np.angle(results_arr))
#Z = np.absolute(results_arr)
Z = np.imag(results_arr)
calc.plot_2d(Y, X,Z, flip = False, cmap = 'Reds')

In [ ]:
# qubit_parameters.update_parameter("qb_ef_freq", qubit_parameters["qb_ef_freq"] + 15e6)

In [ ]:
Data_ef_rabi = {'rabi_shev_ef': results_arr,
               'detuning_ef': detuning_ef,
               'rabi_ef_amp': rabi_ef_amp,
               }

Data_ef_rabi.update(qubit_parameters._params)
Data_ef_rabi.update(lo_settings._params)

file_name = 'Rabi_shev_ef'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_ef_rabi)

In [ ]:
file_name = 'Qubit_params'
file_path = get_path_to_file(file_name, '.txt', sample_parameters)
with open(file_path, 'w') as fp:
    js.dump(qubit_parameters,fp)

### 3.2.4 T1 Experiment

In [ ]:
# delay range for ramsey pulses
t1_min = 0.0
t1_max = 5e-6 #5 * qubit_parameters["qb_t1"]
# how many delay points to sweep
t1_num = 31

# set up sweep parameter
t1_ef_sweep = LinearSweepParameter(uid="t1_delay", start=t1_min, stop=t1_max, count=t1_num)

# how many averages per point: 2^n_average
n_average = 13

In [ ]:
exp_T1_ef = create_T1_ef_1(t1_ef_sweep, 
                           x180_ef, 
                           readout_low, 
                           n_average, 
                           x180, 
                           ge_amp = 1.0)

exp_T1_ef_comp = my_session.compile(exp_T1_ef)

#show_pulse_sheet("Decay-ef", exp_T1_ef_comp)

In [ ]:
T1_ef_results = my_session.run(exp_T1_ef_comp)

In [ ]:
# get measurement data returned by the instruments
t1_ef_res = T1_ef_results.get_data("q0_t1_ef")

# define amplitude axis from qubit parameters
t1_ef_delay = T1_ef_results.get_axis("q0_t1_ef")[0]

In [ ]:
# plot measurement results
fig = plt.figure()
plt.plot(t1_ef_delay, abs(t1_ef_res), ".k")
# plt.plot(t1_ef_delay, np.unwrap(np.angle(t1_ef_res)), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("delay (s)")
plt.title('Decay on ef-transition')

# increase number of plot points for smooth plotting of fit results
delay_plot = np.linspace(t1_ef_delay[0], t1_ef_delay[-1], 5 * len(t1_ef_delay))

# fit measurement results to decaying exponential curve
popt, pcov = fit_T1_ef(x = t1_ef_delay, 
                       # y = np.unwrap(np.angle(t1_ef_res)),
                       y = abs(t1_ef_res),
                       rate1 = 1/4e-6, 
                       rate2 = 1/16e-6,
                       off = 1.54, 
                       A = 0.1,
                       B = 0.1,
                       plot=False)


# plot fit results together with measurement data
plt.plot(delay_plot, func_two_exp(delay_plot, *popt), "-r")
plt.figtext(0.6, 0.8, r'$T_1^{ef}$ = '+'{:.1f}'.format(1/popt[0]*1e6)+r'$\mu$s')
plt.figtext(0.6, 0.7, r'$T_1^{ge}$ = '+'{:.1f}'.format(1/popt[1]*1e6)+r'$\mu$s')

# update qubit parameters - here relaxation time / qubit lifetime
qubit_parameters["qb_ef_t1"] = 1 / popt[0]


#gamma_fe = (popt[0]+popt[1])
#gamma_ge = popt[0]*popt[1]/gamma_fe

print('T1_fe:', 1/popt[0])
print('T1_ge:', 1/popt[1])

#save figure
figname = 'T1_ef_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
Data_T1_ef = {'t1_ef_res': t1_res,
            't1_ef_delay': t1_delay
            }

Data_T1_ef.update(qubit_parameters)
Data_T1_ef.update(lo_settings)

file_name = 'T1_ef_onege_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_T1_ef)

### 3.2.5 Ramsey Experiment

In [ ]:
# qubit_parameters['ramsey_det_ef'] = 2e6
qubit_parameters.update_parameter('ramsey_det_ef', 0.5e6)

In [ ]:
# delay range for ramsey pulses
ramsey_min = 0.0
#ramsey_max = 4.0 * qubit_parameters["ramsey_det"]
ramsey_max = 10e-6 #3.5 * qubit_parameters["qb_t2_ramsey"]
#ramsey_max = 1.5e-6 
# how many delay points to sweep
#ramsey_num = 151
ramsey_num = 401

# set up delay sweep parameter
ramsey_ef_sweep = LinearSweepParameter(
    uid="ramsey_delay", start=ramsey_min, stop=ramsey_max, count=ramsey_num
)

# how many averages per point: 2^n_average
n_average = 11

In [ ]:
#make experiment
exp_ramsey_ef_2 = make_ramsey_ef_2(ramsey_ef_sweep, 
                 x90_ef, 
                 readout_pulse, 
                 readout_weighting_function, 
                 qubit_parameters["relax"], 
                 n_average, 
                 x180, 
                 ge_amp = 1.0)

#set signal map
exp_ramsey_ef_2.set_signal_map(qubit_ef_map)

#make and set calibration
ramsey_ef_cal = make_ramsey_ef_calib(qubit_parameters["qb_ef_freq"], qubit_parameters['ramsey_det_ef'])
exp_ramsey_ef_2.set_calibration(ramsey_ef_cal) 

#compile the experiment
exp_ramsey_ef_2_comp = my_session.compile(exp_ramsey_ef_2)

#show_pulse_sheet("Ramsey-ef", exp_ramsey_ef_2_comp)

In [ ]:
#run the experiment
ramsey_ef_results = my_session.run(exp_ramsey_ef_2_comp)

In [ ]:
# get measurement data returned by the instruments
ramsey_ef_res = ramsey_ef_results.get_data("q0_ramsey_ef")

# define time axis from qubit parameters
ramsey_ef_delay = ramsey_ef_results.get_axis("q0_ramsey_ef")[0]

In [ ]:
# plot measurement results
fig = plt.figure(figsize=(12,8))

data_ramsey_ef = ramsey_ef_res.real
# data_ramsey_ef = np.unwrap(np.angle(ramsey_ef_res))

#plt.plot(ramsey_delay, abs(ramsey_res), ".k") #np.unwrap(np.angle(
plt.plot(ramsey_ef_delay, data_ramsey_ef, ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("delay (s)")
plt.title('Ramsey oscillations')

# increase number of plot points for smooth plotting of fit results
delay_plot = np.linspace(ramsey_ef_delay[0], ramsey_ef_delay[-1], 5 * len(ramsey_ef_delay))

## fit measurement data to decaying sinusoidal oscillatio
popt, pcov = fit_Ramsey(
    ramsey_ef_delay,
    data_ramsey_ef,
    1e6, #qubit_parameters["ramsey_det"],
    0,
    #2 / qubit_parameters["qb_t2_ramsey"],
    2 / 6e-7,
    amp=0.01,
    off=-0.08,
    plot=False,
#     bounds=[
#         [0.1e6, -np.pi, 0.05 / qubit_parameters["qb_t2_ramsey"], 0.0001, -2],
#         [50e6, np.pi, 10 / qubit_parameters["qb_t2_ramsey"], 2, 2],
#     ],
    bounds=[
        [0.01e6, -np.pi, 0.05 / 6e-7, 0.0001, -4],
        [150e6, np.pi, 30 / 6e-7, 2, 4],
    ],
)
#print(popt)

# plot fit results together with experimental data
plt.plot(delay_plot, func_decayOsc(delay_plot, *popt), "-r")
plt.figtext(0.6, 0.8, 'T2 = '+'{:.3f}'.format(1/popt[2]*1e6)+'us')
plt.figtext(0.6, 0.75, r'$\Delta$ = '+'{:.3f}'.format(popt[0]/(2*np.pi)*1e-6)+'MHz')

# update qubit parameters - here qubit dephasing time and qubit frequency
#qubit_parameters["qb_t2_ramsey"] = 1 / popt[2]

# qubit_parameters.update_parameter('qb_ef_freq', qubit_parameters['qb_ef_freq'] + qubit_parameters["ramsey_det_ef"] - popt[0]/np.pi/2)

# T2 Ramsey error in us
err = np.sqrt(np.diag(pcov))
#print(f'T2* = {1/popt[2]*1e6:.3f} +- {err[2]/popt[2]*qubit_parameters["qb_t2_ramsey"]*1e6:.3f} us')
print(f'Detuning is {popt[0]/np.pi/2*1e-6:.4} +- {err[0]/np.pi/2*1e-6:.2} MHz')

#save figure
figname = 'Ramsey_ef_slice_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
Data_Ramsey_ef = {'ramsey_ef_res': ramsey_ef_res,
            'ramsey_ef_delay': ramsey_ef_delay
            }

Data_Ramsey_ef.update(qubit_parameters._params)
Data_Ramsey_ef.update(lo_settings._params)

file_name = 'Ramsey_ef_onege_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_Ramsey_ef)